In [2]:
# ============================================================
# Cell 1 — Install dependencies
# ============================================================
!pip install -q transformers accelerate snac soundfile

In [3]:

import torch
import numpy as np
import soundfile as sf
from transformers import AutoTokenizer, AutoModelForCausalLM
from snac import SNAC

MODEL_ID = "kenpath/svara-tts-v1"

# Orpheus-family special token IDs (svara-tts-v1 shares this scheme)
END_OF_TEXT     = 128009
START_OF_SPEECH = 128257
END_OF_SPEECH   = 128258
START_OF_HUMAN  = 128259
END_OF_HUMAN    = 128260
AUDIO_TOKEN_LO  = 128266
AUDIO_TOKEN_HI  = 128266 + 7 * 4096

In [5]:

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    dtype=torch.bfloat16,
    device_map="cuda",
)
model.eval()

Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

LlamaForCausalLM(
  (model): LlamaModel(
    (embed_tokens): Embedding(156940, 3072, padding_idx=128263)
    (layers): ModuleList(
      (0-27): 28 x LlamaDecoderLayer(
        (self_attn): LlamaAttention(
          (q_proj): Linear(in_features=3072, out_features=3072, bias=False)
          (k_proj): Linear(in_features=3072, out_features=1024, bias=False)
          (v_proj): Linear(in_features=3072, out_features=1024, bias=False)
          (o_proj): Linear(in_features=3072, out_features=3072, bias=False)
        )
        (mlp): LlamaMLP(
          (gate_proj): Linear(in_features=3072, out_features=8192, bias=False)
          (up_proj): Linear(in_features=3072, out_features=8192, bias=False)
          (down_proj): Linear(in_features=8192, out_features=3072, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): LlamaRMSNorm((3072,), eps=1e-05)
        (post_attention_layernorm): LlamaRMSNorm((3072,), eps=1e-05)
      )
    )
    (norm): LlamaRMSNorm((3072

In [6]:
# ============================================================
# Cell 4 — Load the SNAC audio decoder (runs fine on CPU)
# ============================================================
snac_model = SNAC.from_pretrained("hubertsiuzdak/snac_24khz").to("cpu")

config.json:   0%|          | 0.00/300 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/79.5M [00:00<?, ?B/s]

In [7]:
# ============================================================
# Cell 5 — Helper: convert generated audio-token codes to a waveform
# ============================================================
def redistribute_codes(code_list):
    layer_1, layer_2, layer_3 = [], [], []
    for i in range(len(code_list) // 7):
        layer_1.append(code_list[7*i])
        layer_2.append(code_list[7*i + 1] - 4096)
        layer_3.append(code_list[7*i + 2] - 2*4096)
        layer_3.append(code_list[7*i + 3] - 3*4096)
        layer_2.append(code_list[7*i + 4] - 4*4096)
        layer_3.append(code_list[7*i + 5] - 5*4096)
        layer_3.append(code_list[7*i + 6] - 6*4096)
    if not layer_1:
        return torch.zeros(1, 1, 12000)  # silent fallback
    clamp = lambda vals: [max(0, min(4095, v)) for v in vals]
    codes = [
        torch.tensor(clamp(layer_1)).unsqueeze(0),
        torch.tensor(clamp(layer_2)).unsqueeze(0),
        torch.tensor(clamp(layer_3)).unsqueeze(0),
    ]
    return snac_model.decode(codes)

In [12]:
# ============================================================
# Cell 6 (updated) — Main synthesize function with seed support
# ============================================================
def synthesize(text, voice="Hindi (Female)", max_new_tokens=1200,
               temperature=0.6, top_p=0.95, repetition_penalty=1.1,
               seed=None):
    if seed is not None:
        torch.manual_seed(seed)

    tagged = f"{voice}: {text}"
    text_ids = tokenizer(tagged, return_tensors="pt").input_ids

    soh = torch.tensor([[START_OF_HUMAN]], dtype=torch.int64)
    end = torch.tensor([[END_OF_TEXT, END_OF_HUMAN]], dtype=torch.int64)
    input_ids = torch.cat([soh, text_ids, end], dim=1).to(model.device)
    attention_mask = torch.ones_like(input_ids)

    generated = model.generate(
        input_ids=input_ids,
        attention_mask=attention_mask,
        max_new_tokens=max_new_tokens,
        do_sample=True,
        temperature=temperature,
        top_p=top_p,
        repetition_penalty=repetition_penalty,
        eos_token_id=END_OF_SPEECH,
    )

    row = generated[0]
    sos_pos = (row == START_OF_SPEECH).nonzero(as_tuple=True)[0]
    if len(sos_pos) > 0:
        row = row[sos_pos[-1].item() + 1:]

    audio_only = row[(row >= AUDIO_TOKEN_LO) & (row < AUDIO_TOKEN_HI)]
    n = (audio_only.size(0) // 7) * 7
    code_list = [t.item() - AUDIO_TOKEN_LO for t in audio_only[:n]]

    waveform = redistribute_codes(code_list)
    return waveform.detach().squeeze().cpu().numpy().astype(np.float32)

In [14]:
# ============================================================
# Cell 7 — Multi-language, multi-emotion synthesis (with seed)
# ============================================================
from IPython.display import Audio, display
import soundfile as sf

SEED = 42 

requests = [
    {
        "lang": "Hindi",
        "text": "नमस्ते, मैं स्वरा हूं। आज मौसम बहुत अच्छा है।",
        "voice": "Hindi (Female)",
        "emotion": "<happy>",
    },
    {
        "lang": "Marathi",
        "text": "नमस्कार, मी स्वरा आहे. आज हवामान खूप छान आहे.",
        "voice": "Marathi (Female)",
        "emotion": "<happy>",
    },
    {
        "lang": "Tamil",
        "text": "வணக்கம், நான் ஸ்வரா. இன்று வானிலை மிகவும் நன்றாக இருக்கிறது.",
        "voice": "Tamil (Female)",
        "emotion": "<happy>",
    },
    {
        "lang": "Telugu",
        "text": "నమస్కారం, నేను స్వర. ఈరోజు వాతావరణం చాలా బాగుంది.",
        "voice": "Telugu (Female)",
        "emotion": "<happy>",
    },
]

for req in requests:
    full_text = f"{req['text']} {req['emotion']}"
    print(f"Generating {req['lang']} ({req['emotion']})...")

    wav = synthesize(full_text, voice=req["voice"], seed=SEED)

    filename = f"output_{req['lang'].lower()}.wav"
    sf.write(filename, wav, 24000)

    print(f"  saved: {filename}")
    display(Audio(wav, rate=24000))

Generating Hindi (<happy>)...
  saved: output_hindi.wav


Generating Marathi (<happy>)...
  saved: output_marathi.wav


Generating Tamil (<happy>)...
  saved: output_tamil.wav


Generating Telugu (<happy>)...
  saved: output_telugu.wav
